# European Regas Slots in the Money

This script is used to look at the Moneyness of currently available European Regas Slots

For a full explanation of how to import our Regas Slots or DES Hub Netbacks datasets, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
# import libraries for importing data
import json
import os
import sys
import pandas as pd
from base64 import b64encode
from urllib.parse import urljoin
from pprint import pprint
import requests
from io import StringIO
import time
import numpy as np
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")


In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Global Date Input

Input which release date to fetch data & generate the slot moneyness table for. This input has 2 available options:
- "latest" = generates the slot moneyness table for the latest data release for all datasets. This is usually the last business day
- "yyyy-mm-dd" = input a date in the 'yyyy-mm-dd' format to choose a custom release date to generate the slot moneyness table for

In [ ]:
date = 'latest'
#date = '2026-06-24'

# 3. Fetching Regas Slots Data

Here we define the functions used to call the Regas Slots data from the Access Regas Slot Availability API. For more information on the Access Regas Slot Availability API, please visit the API website or download our Access Regas Slot API code sample:

https://www.sparkcommodities.com/api/lng-access/slot-availability.html

In [ ]:
# Fetchinbg reference data for slots

def fetch_slots_reference_data(access_token, format='csv'):

    uri = "/beta/terminal-slots/terminals/"
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

    return data

In [ ]:
# Call reference-data function
slots_ref = fetch_slots_reference_data(access_token, format='csv')
slots_ref.head(5)

In [ ]:
# Fetching slot availability data for defined date

def fetch_slots(access_token, date, format='csv'):

    uri = "/beta/terminal-slots/releases/{}".format(date)
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

    return data

In [ ]:
# Calling the above function
slots_df = fetch_slots(access_token, date=date, format='csv')
slots_df['ReleaseDate'] = pd.to_datetime(slots_df['ReleaseDate'])
slots_df.head()

# 4. Fetching & Computing Moneyness DataFrame

## 4.1. Fetching DES Hub Netbacks data

Here we define the functions used to call the DES Hub Netbacks data from the Access DES Hub Netbacks API. For more information on the Access DES Hub Netbacks API, please visit the API website or download our Access DES Hub Netbacks code sample:

https://www.sparkcommodities.com/api/lng-access/des-hub-netbacks.html

In [ ]:
## Defining the reference-data calling function

def fetch_deshub_reference_data(access_token, format='csv'):

    uri = "/v1.0/lng/access/des-hub-netbacks/reference-data/"
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

    return data


# fetching in CSV format (easier to browse the terminal list)
ref_data = fetch_deshub_reference_data(access_token, format='csv')
ref_data

In [ ]:
## Defining the function

def fetch_historical_deshub_release(access_token, unit, start=None, end=None,
                                    terminal_uuid=None, limit=None, offset=None,
                                    format='csv'):

    query_params = "?unit={}".format(unit)

    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    if terminal_uuid is not None:
        query_params += "&terminal-uuid={}".format(terminal_uuid)
    if limit is not None:
        query_params += "&limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    uri = "/v1.0/lng/access/des-hub-netbacks/{}".format(query_params)
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

            for c in list(data.columns)[8:]:
                data[c] = pd.to_numeric(data[c], errors='coerce')
            data['ReleaseDate'] = pd.to_datetime(data['ReleaseDate'])
            data['DeliveryMonth'] = pd.to_datetime(data['DeliveryMonth'])

    return data

In [ ]:
# Generating start and end dates

if date == 'latest':
    end = datetime.datetime.now().strftime('%Y-%m-%d')
    prev_bday = (datetime.datetime.now() - pd.offsets.BDay(1)).strftime('%Y-%m-%d')
else:
    end = date
    prev_bday = (datetime.datetime(end) - pd.offsets.BDay(1)).strftime('%Y-%m-%d')

# Call Data
deshub_df = fetch_historical_deshub_release(access_token, unit='usd-per-mmbtu',
                                start=prev_bday, end=end, format='csv')

deshub_df = deshub_df[deshub_df['ReleaseDate'] == deshub_df['ReleaseDate'].max()]

deshub_df.head(5)

In [ ]:
# Global sunk slot components - added back for every terminal
sunk_slot_cols = ['SlotLtCapacityEstimate', 'SlotUnloadStorageRegas', 'SlotBerth', 'SlotBerthUnloadStorageRegas']

# Per-terminal extra sunk costs - layered on top of the global sunk costs defined above
# Value is a list of cost columns to treat as sunk for that terminal only
# Edit this dict to suit your analysis. Unlisted terminals use the global set.

sunk_terminal_costs = {
    # 'Isle of Grain': ['EntryCapacity', 'EntryVariable'],
    # 'South Hook': ['EntryCapacity'],
}

# Sink global sunk costs across all terminals
sunk_add_back = deshub_df[sunk_slot_cols].fillna(0).sum(axis=1)

# Sink terminal-specific sunk costs
for terminal, cols in sunk_terminal_costs.items():
    if not cols:
        continue
    missing = [c for c in cols if c not in deshub_df.columns]
    if missing:
        raise KeyError('sunk_terminal_costs[{!r}] references missing columns: {}'.format(terminal, missing))
    mask = deshub_df['TerminalName'] == terminal
    if mask.any():
        sunk_add_back.loc[mask] += deshub_df.loc[mask, cols].fillna(0).sum(axis=1)

deshub_df['NetbackTTFBasisSunk'] = deshub_df['NetbackTTFBasis'] + sunk_add_back
deshub_df['NetbackOutrightSunk'] = deshub_df['NetbackOutright'] + sunk_add_back

deshub_df[['ReleaseDate', 'TerminalName', 'DeliveryMonthIndex', 'NetbackTTFBasis',
         'NetbackTTFBasisSunk']].head(10)


In [ ]:
# one row per (ReleaseDate, DeliveryMonth), one column per terminal
terminal_list = list(deshub_df['TerminalName'].unique())

id_cols = ['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']
value_col = 'NetbackTTFBasisSunk'

deshub_final = (
    deshub_df
    .pivot_table(index=id_cols, columns='TerminalName', values=value_col, aggfunc='first')
    .reset_index()
)
deshub_final.columns.name = None

# restore terminal-appearance ordering (pivot_table sorts alphabetically by default)
deshub_final = deshub_final[id_cols + terminal_list]
deshub_final.head(5)

## 4.2. Fetching Cargo Prices

Here we define the functions used to call the SparkNWE/SWE DES LNG data from the Cargo Contracts API. For more information on the Cargo Contracts API, please visit the API website or download our Cargo Contracts code sample:

https://www.sparkcommodities.com/api/lng-cargo/contracts.html

In [ ]:
# Defining function to call contracts endpoint to import cargo data
def fetch_cargo_releases(access_token, ticker, limit=4, offset=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    uri = "/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params)
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token)

    return content['data']


# call front month and forward curve, combine into one DataFrame per release date
def cargo_focurve(access_token, ticker, limit):

    f_tick = ticker + '-b-f'
    fo_tick = ticker + '-b-fo'

    f_data = fetch_cargo_releases(access_token, f_tick, limit)
    fo_data = fetch_cargo_releases(access_token, fo_tick, limit)

    release_dates = []
    period_start = []
    contract_ids = []
    spark = []

    # single-release convenience path (latest only)
    if limit == 1:
        release_date = f_data[0]["releaseDate"]
        contract_ids.append(f_data[0]['contractId'])
        release_dates.append(release_date)

        period_start_at = f_data[0]['data'][0]['dataPoints'][0]["deliveryPeriod"]["startAt"]
        period_start.append(period_start_at)
        spark.append(f_data[0]['data'][0]['dataPoints'][0]['derivedPrices']['usdPerMMBtu']['spark'])

        for release in fo_data:
            for data_point in release['data'][0]['dataPoints']:
                contract_ids.append(release['contractId'])
                release_dates.append(release["releaseDate"])
                period_start.append(data_point["deliveryPeriod"]["startAt"])
                spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])

    # full historical path: align each front month release with the most recent
    # forward curve release on or before it
    else:
        cl = 0

        for l in range(limit):
            try:
                release_date = f_data[l]["releaseDate"]
            except (IndexError, KeyError):
                break

            contract_ids.append(f_data[l]['contractId'])
            release_dates.append(release_date)
            release_datetime = datetime.datetime.strptime(release_date, '%Y-%m-%d')

            front_period_start_at = f_data[l]['data'][0]['dataPoints'][0]["deliveryPeriod"]["startAt"]
            period_start.append(front_period_start_at)
            spark.append(f_data[l]['data'][0]['dataPoints'][0]['derivedPrices']['usdPerMMBtu']['spark'])

            # guard against running off the end of fo_data
            if cl >= len(fo_data):
                continue

            fo_datetime = datetime.datetime.strptime(fo_data[cl]['releaseDate'], '%Y-%m-%d')

            # assessment date - front month AND forward curve both published today
            if release_datetime == fo_datetime:
                for data_point in fo_data[cl]['data'][0]['dataPoints']:
                    contract_ids.append(fo_data[cl]['contractId'])
                    release_dates.append(release_date)
                    period_start.append(data_point["deliveryPeriod"]["startAt"])
                    spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])
                cl += 1

            # non-assessment date - persist most recent published forward curve
            # but drop the data point that overlaps with today's front month
            else:
                for data_point in fo_data[cl]['data'][0]['dataPoints']:
                    period_start_at = data_point["deliveryPeriod"]["startAt"]
                    if period_start_at != front_period_start_at:
                        contract_ids.append(fo_data[cl]['contractId'])
                        release_dates.append(release_date)
                        period_start.append(period_start_at)
                        spark.append(data_point['derivedPrices']['usdPerMMBtu']['spark'])

    hist_cargo_df = pd.DataFrame({
        'Release Date': release_dates,
        'ticker': contract_ids,
        'Period Start': period_start,
        'Price': spark,
    })

    hist_cargo_df['Price'] = pd.to_numeric(hist_cargo_df['Price'])
    hist_cargo_df['Release Date'] = pd.to_datetime(hist_cargo_df['Release Date']).dt.tz_localize(None)

    return hist_cargo_df

In [ ]:
# calling all historical data
nwe_fo_df = cargo_focurve(access_token, 'sparknwe', limit=5000)
swe_fo_df = cargo_focurve(access_token, 'sparkswe', limit=5000)

In [ ]:
# Merging the NWE and SWE datasets into a single dataframe
cargo_df = pd.merge(nwe_fo_df, swe_fo_df, on=['Release Date', 'Period Start'], how='left')
cargo_df = cargo_df.rename(columns={
    'Price_x': 'SparkNWE',
    'Price_y': 'SparkSWE',
})
cargo_df = cargo_df.drop(columns=['ticker_x', 'ticker_y'])

# cut off data prior to daily front-month assessments for NWE (switchover 2023-11-23)
switch_date = '2023-11-23'
date_limit = cargo_df[cargo_df['Release Date'] == switch_date].index[-1]
cargo_df = cargo_df.iloc[:date_limit + 1].copy()

# Persist SparkSWE forward curves across non-assessment days (SWE fo is weekly, NWE is daily) 
# Groupby 'Period Start' to make sure data fills match against Period Start
# limit=14 covers potential bank holiday gaps.
cargo_df['Period Start'] = pd.to_datetime(cargo_df['Period Start'])
cargo_df = cargo_df.sort_values(['Period Start', 'Release Date']).reset_index(drop=True)
cargo_df['SparkSWE'] = cargo_df.groupby('Period Start')['SparkSWE'].ffill(limit=14)

# Order by release date
cargo_df = cargo_df.sort_values(['Release Date', 'Period Start'], ascending=[False, True]).reset_index(drop=True)

## 4.3. Moneyness Calculations

Moneyness is calculated as the difference between DES Hub Netbacks and SparkNWE/SWE markers. For more Moneyness Analytics, visit our other Analytics Gallery code samples:

https://www.sparkcommodities.com/api/code-examples/analytics-examples.html

In [ ]:
# Region mapping 
terminal_region_dict = {
    'Gate': 'nwe',
    'Isle of Grain': 'nwe',
    'Zeebrugge': 'nwe',
    'South Hook': 'nwe',
    'Dunkerque': 'nwe',
    'Le Havre': 'nwe',
    'Montoir': 'nwe',
    'EemsEnergyTerminal': 'nwe',
    'Brunsbuttel': 'nwe',
    'Deutsche Ostsee': 'nwe',
    'Deutsche Ostsee Phase 1': 'nwe',
    'Wilhelmshaven 1': 'nwe',
    'Wilhelmshaven 2': 'nwe',
    'Stade': 'nwe',
    'Fos Cavaou': 'swe',
    'Adriatic': 'swe',
    'OLT Toscana': 'swe',
    'Piombino': 'swe',
    'Ravenna': 'swe',
    'Spain TVB': 'swe',
}

# Country mapping (optional)
terminal_country_dict = {
    'Netherlands': ['Gate', 'EemsEnergyTerminal'],
    'UK':          ['Isle of Grain', 'South Hook', 'Dragon'],
    'Belgium':     ['Zeebrugge'],
    'France':      ['Dunkerque', 'Le Havre', 'Montoir', 'Fos Cavaou'],
    'Germany':     ['Brunsbuttel', 'Deutsche Ostsee',
                    'Wilhelmshaven 1', 'Wilhelmshaven 2', 'Stade'],
    'Italy':       ['Adriatic', 'OLT Toscana', 'Piombino', 'Ravenna'],
    'Spain':       ['Spain TVB'],
    'Finland':     ['Inkoo'],
    'Lithuania':   ['Klaipeda'],
}

# Reverse Mapping
terminal_to_country = {t: c for c, terms in terminal_country_dict.items() for t in terms}

In [ ]:
# Vectorised moneyness calculation.

# only consider terminals mapped to a region
new_terminal_list = [t for t in terminal_list if t in terminal_region_dict]
unmapped = [t for t in terminal_list if t not in terminal_region_dict]
if unmapped:
    print('Skipping terminals not mapped to a region in terminal_region_dict: {}'.format(unmapped))

# merge DES Hub netbacks with NWE/SWE data 
money_merge = pd.merge(
    deshub_final,
    cargo_df[['Release Date', 'Period Start', 'SparkNWE', 'SparkSWE']],
    left_on=['ReleaseDate', 'DeliveryMonth'],
    right_on=['Release Date', 'Period Start'],
    how='left',
)

# initialise output
money_df = money_merge[['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']].copy()

# compute moneyness per terminal
nwe_terms = [t for t in new_terminal_list if terminal_region_dict[t] == 'nwe']
swe_terms = [t for t in new_terminal_list if terminal_region_dict[t] == 'swe']

# Subtract either NWE or SWE marker depending on terminal
for t in nwe_terms:
    money_df[t] = money_merge[t] - money_merge['SparkNWE']
for t in swe_terms:
    money_df[t] = money_merge[t] - money_merge['SparkSWE']

# round to 3 d.p.
money_df[new_terminal_list] = money_df[new_terminal_list].round(3)

# 5. Generating final "Slots In the Money" Dataframe 

In [ ]:
# Preview Moneyness Dataframe
money_df.head(3)

In [ ]:
# Preview Slots Dataframe
slots_df.head(3)

In [ ]:
# Matching money_df to the slots_df format

# create the same "M+X" column names as slots_df
format_cols = [f'M+{i}' for i in range(1, 13)]
id_cols = ['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']
money_terminals = [c for c in money_df.columns if c not in id_cols]

# remove release date and index by terminal
slots_display = slots_df.set_index('TerminalName').drop(columns='ReleaseDate')

# 1. reshape money_df to match approximate structure of slots_df
money_lookup = (money_df.set_index('DeliveryMonthIndex')[money_terminals].T)
money_lookup.index.name = 'TerminalName'

# creating an exact mapping of Moneyness values to slots_df, so it can be referenced for conditonal formatting
gmap = (
    money_lookup
    .reindex(index=slots_display.index)
    .reindex(columns=format_cols)
    .apply(pd.to_numeric, errors='coerce')
)

gmap.head(3)

# 5.1. Final Table

In [ ]:
# reformat slots_df by dropping release date and indexing by TerminalName
slots_display = slots_df.set_index('TerminalName').drop(columns='ReleaseDate')

# Creating MultiIndex so that table is organised into Countries and Terminals
slots_display['Country'] = slots_display.index.map(terminal_to_country).fillna('Other')
slots_display = (slots_display
                 .set_index('Country', append=True)
                 .reorder_levels(['Country', 'TerminalName'])
                 .sort_index())

# aligning gmap to Country/Terminal MultiIndex
gmap = gmap.reindex(slots_display.index.get_level_values('TerminalName'))
gmap.index = slots_display.index

# creating colourmap params for the table conditional formatting
vmax = np.nanmax(np.abs(gmap.values))        # finding max moneyness value to scale colourmap
gamma = 0.5                                  # colour change rate: <1 = faster colour change near 0
t = (gmap / vmax).clip(-1, 1)                # normalising colourmap values
gmap_bent = np.sign(t) * np.abs(t) ** gamma  # applying colour change rate to the colourmap

# Creating list of columns to show in table
display_cols = ['Total', 'M+0'] + format_cols

# Creating Styled DataFrame of available slots on chosen release date + moneyness
slot_money = (
    slots_display[display_cols].style
    .background_gradient(
        cmap='coolwarm',        # Choose colourmap 
        subset=format_cols,     # which columns to apply conditional formatting to
        gmap=gmap_bent,         # apply colour mapping params 
        axis=None,
        vmin=-1, vmax=1,        
    )
    .format(precision=0)        # how many decimal points
)

print("Slots Moneyness - " + date)

slot_money

# 5.2. Optional Formatting using 'plottable'

This section uses the Python package 'plottable' to apply style formatting to the above table - this is the formatting used in the Analytics Gallery thumbnail. 'plottable' is not a standard Python package and needs to be downloaded separately - as such, the code below is commented out to avoid script breaks. 

For more information on 'plottable', please refer to their documentation:

https://plottable.readthedocs.io/en/latest/index.html#

In [ ]:
"""import matplotlib.pyplot as plt
import matplotlib
from plottable import ColumnDefinition, Table

# reformat slots_df by dropping release date and indexing by TerminalName
slots_display2 = slots_df.set_index('TerminalName').drop(columns='ReleaseDate')

# Adding Country column and sorting accordingly
slots_display2['Country'] = slots_display2.index.map(terminal_to_country).fillna('Other')
slots_display2 = (slots_display2.reset_index()
                 .sort_values(['Country', 'TerminalName'])
                 .reset_index(drop=True))

# blank repeated country labels so it reads like a grouped index
slots_display2['Country'] = slots_display2['Country'].mask(
    slots_display2['Country'].duplicated(), '')
slots_display2 = slots_display2.set_index('Country')   # Country becomes the first column

# moneyness lookup keyed by TerminalName & ignoring all terminals without Moneyness values
id_cols = ['ReleaseDate', 'DeliveryMonth', 'DeliveryMonthIndex', 'DeliveryMonthName']
money_terminals = [c for c in money_df.columns if c not in id_cols]
gmap_plot = (
    money_df.set_index('DeliveryMonthIndex')[money_terminals].T
    .reindex(index=slots_df['TerminalName'].unique())   # add slot terminals missing from money -> NaN rows
    .reindex(columns=format_cols)
    .apply(pd.to_numeric, errors='coerce')
)

# defining colourmap with non-linear scaling: steeper colour change near zero
rg_cmap = matplotlib.colormaps['coolwarm_r']    # chosen colourmap
vmax = np.nanmax(np.abs(gmap_plot.values))      # finding max moneyness value to scale colourmap
gamma = 0.5                                     # colour change rate: <1 = faster colour change near 0

# defining function to create non-linear colourmap scaling
def moneyness_color(v):
    t = np.clip(v / vmax, -1, 1)          # linear normalise to [-1, 1]
    t = np.sign(t) * np.abs(t) ** gamma   # bend the curve around 0
    return rg_cmap((t + 1) / 2)           # map [-1, 1] -> [0, 1] -> colour

# Building Table
fig, ax = plt.subplots(figsize=(30, 12))

# creating table columns & associated parameters
col_defs = (
    [ColumnDefinition(name='Country',
                      textprops={'ha': 'left', 'weight': 'bold'},
                      width=1.0, border='right')]
    + [ColumnDefinition(name='TerminalName',
                        textprops={'ha': 'left', 'weight': 'bold'},
                        width=1.6, border='right')]
    + [ColumnDefinition(name='Total',
                        textprops={'ha': 'center', 'weight': 'bold'}, border='right')]
    + [ColumnDefinition(name='M+0',
                        textprops={'ha': 'center'}, border='right')]
    + [ColumnDefinition(name=c, title=c, textprops={'ha': 'center'},
                        group='Slots Moneyness - ' + date) for c in format_cols]
)

# creating table
table = Table(
    slots_display2[['TerminalName', 'Total', 'M+0'] + format_cols],
    ax=ax,
    column_definitions=col_defs,
    row_dividers=True,
    col_label_divider=True,
    footer_divider=True,
    textprops={'fontsize': 12},
    row_divider_kw={'linewidth': 1, 'linestyle': (0, (1, 5))},
    col_label_divider_kw={'linewidth': 1, 'linestyle': '-'},
    column_border_kw={'linewidth': 1, 'linestyle': '-'},
)

# Styling cells: bold non-zero counts & colour M+1..M+12 by moneyness
color_cols = set(format_cols)
for c in ['Total', 'M+0'] + format_cols:
    for cell in table.columns[c].cells:
        # bold any non-zero slot count
        if isinstance(cell.content, (int, float, np.integer, np.floating)) and cell.content != 0:
            cell.text.set_fontweight('bold')
        # colour the delivery-month cells from the moneyness gmap
        if c in color_cols:
            val = gmap_plot.loc[slots_display2['TerminalName'].iloc[cell.row_idx], c]
            if pd.notna(val):
                cell.rectangle_patch.set_facecolor(moneyness_color(val))

plt.show()"""